In [1]:
import time
import random
import pandas as pd

from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager


districts = [

    "Araria",
    "Arwal",
    "Aurangabad",
    "Banka",
    "Begusarai",
    "Bhagalpur",
    "Bhojpur",
    "Buxar",
    "Darbhanga",
    "East Champaran",
    "Gaya",
    "Gopalganj",
    "Jamui",
    "Jehanabad",
    "Kaimur",
    "Katihar",
    "Khagaria",
    "Kishanganj",
    "Lakhisarai",
    "Madhepura",
    "Madhubani",
    "Munger",
    "Muzaffarpur",
    "Nalanda",
    "Nawada",
    "Patna",
    "Purnia",
    "Rohtas",
    "Saharsa",
    "Samastipur",
    "Saran",
    "Sheikhpura",
    "Sheohar",
    "Sitamarhi",
    "Siwan",
    "Supaul",
    "Vaishali",
    "West Champaran"

]


search_types = [
    "hospitals"
]


options = webdriver.ChromeOptions()

options.add_argument("--start-maximized")

options.add_argument(
    "--disable-blink-features=AutomationControlled"
)

options.add_argument("--disable-notifications")


driver = webdriver.Chrome(

    service=Service(
        ChromeDriverManager().install()
    ),

    options=options
)


driver.get("https://www.google.com/maps")

print("Opening Google Maps...")

time.sleep(15)

print("Google Maps Loaded")


all_data = []

visited_links = set()


def get_search_box():

    selectors = [

        '//input[@id="searchboxinput"]',
        '//input[contains(@placeholder,"Search")]',
        '//input[contains(@aria-label,"Search")]',
        '//input'

    ]

    for selector in selectors:

        try:

            box = driver.find_element(
                By.XPATH,
                selector
            )

            return box

        except:
            pass

    return None


def extract_school_data():

    try:

 
        try:
            name = driver.find_element(
                By.TAG_NAME,
                "h1"
            ).text
        except:
            name = ""

        
        try:
            address = driver.find_element(
                By.XPATH,
                '//button[contains(@data-item-id,"address")]'
            ).text
        except:
            address = ""

        try:
            phone = driver.find_element(
                By.XPATH,
                '//button[contains(@data-item-id,"phone")]'
            ).text
        except:
            phone = ""

        try:
            website = driver.find_element(
                By.XPATH,
                '//a[contains(@data-item-id,"authority")]'
            ).get_attribute("href")
        except:
            website = ""

        try:
            rating = driver.find_element(
                By.XPATH,
                '//div[@role="img"]'
            ).get_attribute("aria-label")
        except:
            rating = ""

        return {

            "Hospital Name": name,
            "Address": address,
            "Phone": phone,
            "Website": website,
            "Rating": rating,
            "Google Maps URL": driver.current_url

        }

    except:

        return None



for district in districts:

    for search_type in search_types:

        try:

            query = (
                f"{search_type} "
                f"in {district} Bihar"
            )

            print(f"\nSearching: {query}")


            search_box = get_search_box()

            if not search_box:

                print("Search Box Not Found")

                continue

            ActionChains(driver).move_to_element(
                search_box
            ).click().perform()

            time.sleep(1)

            search_box.send_keys(
                Keys.CONTROL + "a"
            )

            time.sleep(1)

            search_box.send_keys(Keys.DELETE)

            time.sleep(1)

   
            for char in query:

                search_box.send_keys(char)

                time.sleep(0.03)

            search_box.send_keys(Keys.ENTER)

            print("Search Submitted")

            time.sleep(8)


            try:

                scrollable_div = driver.find_element(
                    By.XPATH,
                    '//div[@role="feed"]'
                )

                previous_count = 0

                stagnant = 0

                for i in range(100):

                    driver.execute_script(
                        """
                        arguments[0].scrollTop =
                        arguments[0].scrollHeight
                        """,
                        scrollable_div
                    )

 
                    time.sleep(4)

                    listings = driver.find_elements(
                        By.XPATH,
                        '//a[contains(@href,"/maps/place/")]'
                    )

                    current_count = len(listings)

                    print(
                        f"{district} | "
                        f"{search_type} | "
                        f"Scroll {i+1} -> "
                        f"{current_count}"
                    )

                    if current_count == previous_count:

                        stagnant += 1

                    else:

                        stagnant = 0

                    if stagnant >= 4:

                        break

                    previous_count = current_count

            except Exception as e:

                print(f"Scrolling Error: {e}")


            listings = driver.find_elements(
                By.XPATH,
                '//a[contains(@href,"/maps/place/")]'
            )

            links = []

            for listing in listings:

                try:

                    link = listing.get_attribute("href")

                    if (
                        link
                        and link not in visited_links
                    ):

                        links.append(link)

                        visited_links.add(link)

                except:
                    pass

            print(f"Collected {len(links)} links")


            for index, link in enumerate(links):

                try:

                    print(
                        f"{district} | "
                        f"{index+1}/{len(links)}"
                    )

                    driver.get(link)

                    time.sleep(
                        random.uniform(2, 4)
                    )

                    data = extract_school_data()

                    if data:

                        data["District"] = district

                        data["Search Type"] = search_type

                        all_data.append(data)

                        print(data["Hospital Name"])



                    if len(all_data) % 100 == 0:

                        backup_df = pd.DataFrame(
                            all_data
                        )

                        backup_df.drop_duplicates(
                            subset=[
                                "Hospital Name",
                                "Google Maps URL"
                            ],
                            inplace=True
                        )

                        backup_filename = (
                            "bihar_backup.csv"
                        )

                        backup_df.to_csv(
                            backup_filename,
                            index=False
                        )

                        print("Backup Saved")

                except Exception as e:

                    print(
                        f"School Error: {e}"
                    )

        except Exception as e:

            print(
                f"District Error: {e}"
            )


df = pd.DataFrame(all_data)

df.drop_duplicates(

    subset=[
        "Hospital Name",
        "Google Maps URL"
    ],

    inplace=True
)



filename = (
    "bihar_hospitals.csv"
)

df.to_csv(
    filename,
    index=False
)

print("\nExtraction Completed")

print(
    f"Total Schools Extracted: {len(df)}"
)

print(f"\nSaved File: {filename}")

driver.quit()

Opening Google Maps...
Google Maps Loaded

Searching: hospitals in Araria Bihar
Search Submitted
Araria | hospitals | Scroll 1 -> 16
Araria | hospitals | Scroll 2 -> 20
Araria | hospitals | Scroll 3 -> 24
Araria | hospitals | Scroll 4 -> 24
Araria | hospitals | Scroll 5 -> 24
Araria | hospitals | Scroll 6 -> 24
Araria | hospitals | Scroll 7 -> 24
Collected 24 links
Araria | 1/24
Mk Umeed Emergency Hospital Araria
Araria | 2/24
Sadar hospital araria
Araria | 3/24
AN PRIME HOSPITAL
Araria | 4/24
N.N. Health & Wellness Clinic
Araria | 5/24
Newlife Surgical Hospital
Araria | 6/24
New Pulse Hospital Araria
Araria | 7/24
City Hospital
Araria | 8/24
Usha Maternity & Women's Health Care Centre
Araria | 9/24
Tabassum Hospital: Obs Gynae & Skincare with Laser
Araria | 10/24
ASM Memorial Hospital
Araria | 11/24
GOVERNMENT HOSPITAL ARARIA
Araria | 12/24
Dr M A Rashid Memorial Polyclinic
Araria | 13/24
H.H.L HOSPITAL ARARIA , BIHAR
Araria | 14/24
Apollo 24 Helth Center Araria
Araria | 15/24
Yogmaya

In [1]:
import time
import random
import pandas as pd

from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager

# ==========================================
# BIHAR DISTRICTS
# ==========================================

districts = [

    "Gaya",
    "Gopalganj",
    "Jamui",
    "Jehanabad",
    "Kaimur",
    "Katihar",
    "Khagaria",
    "Kishanganj",
    "Lakhisarai",
    "Madhepura",
    "Madhubani",
    "Munger",
    "Muzaffarpur",
    "Nalanda",
    "Nawada",
    "Patna",
    "Purnia",
    "Rohtas",
    "Saharsa",
    "Samastipur",
    "Saran",
    "Sheikhpura",
    "Sheohar",
    "Sitamarhi",
    "Siwan",
    "Supaul",
    "Vaishali",
    "West Champaran"

]

# ==========================================
# SEARCH TYPES
# ==========================================

search_types = [
    "hospitals"
]

# ==========================================
# CHROME OPTIONS
# ==========================================

options = webdriver.ChromeOptions()

options.add_argument("--start-maximized")

options.add_argument(
    "--disable-blink-features=AutomationControlled"
)

options.add_argument("--disable-notifications")

# ==========================================
# START DRIVER
# ==========================================

driver = webdriver.Chrome(

    service=Service(
        ChromeDriverManager().install()
    ),

    options=options
)

# ==========================================
# OPEN GOOGLE MAPS
# ==========================================

driver.get("https://www.google.com/maps")

print("Opening Google Maps...")

time.sleep(15)

print("Google Maps Loaded")

# ==========================================
# STORAGE
# ==========================================

all_data = []

visited_links = set()

# ==========================================
# SEARCH BOX FUNCTION
# ==========================================

def get_search_box():

    selectors = [

        '//input[@id="searchboxinput"]',
        '//input[contains(@placeholder,"Search")]',
        '//input[contains(@aria-label,"Search")]',
        '//input'

    ]

    for selector in selectors:

        try:

            box = driver.find_element(
                By.XPATH,
                selector
            )

            return box

        except:
            pass

    return None

# ==========================================
# EXTRACT SCHOOL DATA
# ==========================================

def extract_school_data():

    try:

        # SCHOOL NAME
        try:
            name = driver.find_element(
                By.TAG_NAME,
                "h1"
            ).text
        except:
            name = ""

        # ADDRESS
        try:
            address = driver.find_element(
                By.XPATH,
                '//button[contains(@data-item-id,"address")]'
            ).text
        except:
            address = ""

        # PHONE
        try:
            phone = driver.find_element(
                By.XPATH,
                '//button[contains(@data-item-id,"phone")]'
            ).text
        except:
            phone = ""

        # WEBSITE
        try:
            website = driver.find_element(
                By.XPATH,
                '//a[contains(@data-item-id,"authority")]'
            ).get_attribute("href")
        except:
            website = ""

        # RATING
        try:
            rating = driver.find_element(
                By.XPATH,
                '//div[@role="img"]'
            ).get_attribute("aria-label")
        except:
            rating = ""

        return {

            "Hospital Name": name,
            "Address": address,
            "Phone": phone,
            "Website": website,
            "Rating": rating,
            "Google Maps URL": driver.current_url

        }

    except:

        return None

# ==========================================
# START EXTRACTION
# ==========================================

for district in districts:

    for search_type in search_types:

        try:

            query = (
                f"{search_type} "
                f"in {district} Bihar"
            )

            print(f"\nSearching: {query}")

            # ==================================
            # SEARCH
            # ==================================

            search_box = get_search_box()

            if not search_box:

                print("Search Box Not Found")

                continue

            ActionChains(driver).move_to_element(
                search_box
            ).click().perform()

            time.sleep(1)

            search_box.send_keys(
                Keys.CONTROL + "a"
            )

            time.sleep(1)

            search_box.send_keys(Keys.DELETE)

            time.sleep(1)

            # TYPE SLOWLY
            for char in query:

                search_box.send_keys(char)

                time.sleep(0.03)

            search_box.send_keys(Keys.ENTER)

            print("Search Submitted")

            time.sleep(8)

            # ==================================
            # SCROLL RESULTS
            # ==================================

            try:

                scrollable_div = driver.find_element(
                    By.XPATH,
                    '//div[@role="feed"]'
                )

                previous_count = 0

                stagnant = 0

                for i in range(100):

                    driver.execute_script(
                        """
                        arguments[0].scrollTop =
                        arguments[0].scrollHeight
                        """,
                        scrollable_div
                    )

                    # SLOWER WAIT
                    time.sleep(4)

                    listings = driver.find_elements(
                        By.XPATH,
                        '//a[contains(@href,"/maps/place/")]'
                    )

                    current_count = len(listings)

                    print(
                        f"{district} | "
                        f"{search_type} | "
                        f"Scroll {i+1} -> "
                        f"{current_count}"
                    )

                    if current_count == previous_count:

                        stagnant += 1

                    else:

                        stagnant = 0

                    # STOP AFTER 4 SAME COUNTS
                    if stagnant >= 4:

                        break

                    previous_count = current_count

            except Exception as e:

                print(f"Scrolling Error: {e}")

            # ==================================
            # GET LINKS
            # ==================================

            listings = driver.find_elements(
                By.XPATH,
                '//a[contains(@href,"/maps/place/")]'
            )

            links = []

            for listing in listings:

                try:

                    link = listing.get_attribute("href")

                    if (
                        link
                        and link not in visited_links
                    ):

                        links.append(link)

                        visited_links.add(link)

                except:
                    pass

            print(f"Collected {len(links)} links")

            # ==================================
            # VISIT EACH SCHOOL
            # ==================================

            for index, link in enumerate(links):

                try:

                    print(
                        f"{district} | "
                        f"{index+1}/{len(links)}"
                    )

                    driver.get(link)

                    time.sleep(
                        random.uniform(2, 4)
                    )

                    data = extract_school_data()

                    if data:

                        data["District"] = district

                        data["Search Type"] = search_type

                        all_data.append(data)

                        print(data["Hospital Name"])

                    # ==================================
                    # BACKUP SAVE
                    # ==================================

                    if len(all_data) % 100 == 0:

                        backup_df = pd.DataFrame(
                            all_data
                        )

                        backup_df.drop_duplicates(
                            subset=[
                                "Hospital Name",
                                "Google Maps URL"
                            ],
                            inplace=True
                        )

                        backup_filename = (
                            "bihar_backup.csv"
                        )

                        backup_df.to_csv(
                            backup_filename,
                            index=False
                        )

                        print("Backup Saved")

                except Exception as e:

                    print(
                        f"School Error: {e}"
                    )

        except Exception as e:

            print(
                f"District Error: {e}"
            )



# ==========================================
# FINAL SAVE
# ==========================================

new_df = pd.DataFrame(all_data)

new_df.drop_duplicates(

    subset=[
        "Hospital Name",
        "Google Maps URL"
    ],

    inplace=True
)

old_file = "bihar_hospitals.csv"

try:

    old_df = pd.read_csv(old_file)

    final_df = pd.concat(
        [old_df, new_df],
        ignore_index=True
    )

    final_df.drop_duplicates(

        subset=[
            "Hospital Name",
            "Google Maps URL"
        ],

        inplace=True
    )

except:

    final_df = new_df

final_df.to_csv(
    old_file,
    index=False
)

print("\nExtraction Completed")

print(
    f"Total Schools Extracted: {len(final_df)}"
)

print(
    f"\nUpdated Existing File: {old_file}"
)

driver.quit()


Opening Google Maps...
Google Maps Loaded

Searching: hospitals in Gaya Bihar
Search Submitted
Gaya | hospitals | Scroll 1 -> 16
Gaya | hospitals | Scroll 2 -> 20
Gaya | hospitals | Scroll 3 -> 28
Gaya | hospitals | Scroll 4 -> 36
Gaya | hospitals | Scroll 5 -> 40
Gaya | hospitals | Scroll 6 -> 48
Gaya | hospitals | Scroll 7 -> 50
Gaya | hospitals | Scroll 8 -> 50
Gaya | hospitals | Scroll 9 -> 50
Gaya | hospitals | Scroll 10 -> 50
Gaya | hospitals | Scroll 11 -> 50
Collected 50 links
Gaya | 1/50
SNM Medica Multi Speciality Hospital - Best Hospital in Gaya
Gaya | 2/50
Arsh Superspeciality Hospital
Gaya | 3/50
SUDHA HOSPITAL
Gaya | 4/50
Vaishnavi Multispecialty Hospital-Best Hospital in Gaya
Gaya | 5/50
Bihar-Max International Hospital Pvt Ltd | Best Hospital in Gaya, Bihar | best hospital in gaya, Bihar India
Gaya | 6/50
kumar Multispeciality Hospital
Gaya | 7/50
Harihar Global Hospital And Research Centre
Gaya | 8/50
New DSR Hospital Private Limited
Gaya | 9/50
Naroma Advance Ortho & 